# 01 — Data Cleaning

Game-On is a semantic search engine: every game gets turned into a text blob and embedded with SBERT, so search quality depends entirely on how clean and information-dense that text is. This notebook explores the raw Steam dataset's problems and fixes them.

Pipeline: **01 cleaning → 02 feature engineering → 03 EDA → 04 modeling**.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import re

## 2. Load the raw data

In [ ]:
CSV_PATH = '../raw_data/steam_games.csv'
CSV_PATH_IMG = '../raw_data/applications.csv'

df = pd.read_csv(CSV_PATH)
df1 = pd.read_csv(CSV_PATH_IMG, low_memory=False)

df.shape

## 3. Exploring what's wrong (before cleaning)

The columns that matter most here are the ones that will end up in the embedding text: `genre`, `popular_tags`, `game_details`, `game_description`. If those are messy or missing, the embeddings — and therefore the search results — suffer.

In [ ]:
# How much of the text that will feed the embeddings is missing?
df[['genre', 'popular_tags', 'game_details', 'game_description']].isna().sum()

In [ ]:
# Raw description: markdown-style asterisks and inconsistent whitespace
df['game_description'].iloc[0]

In [ ]:
# Price and release_date come in as free-form strings, not numbers/dates
df[['original_price', 'release_date']].drop_duplicates().sample(10, random_state=1)

In [ ]:
# Review data is buried inside a sentence instead of being its own column
df['all_reviews'].dropna().iloc[0]

In [ ]:
# Any exact duplicate games?
df.duplicated(subset=['name', 'url']).sum()

## 4. Cleaning, step by step

**Reviews** — extract the review count and percentage out of the free-text `all_reviews` column

In [ ]:
df['total_review'] = df['all_reviews'].str.extract(r'\(([\d,]+)\)').iloc[:, 0].str.replace(',', '')
df['total_review'] = pd.to_numeric(df['total_review'], errors='coerce')
df['review_per'] = df['all_reviews'].str.extract(r'(\d+)%').iloc[:, 0]
df['review_per'] = pd.to_numeric(df['review_per'], errors='coerce')

**Drop columns** that don't add value for search or recommendation

In [ ]:
df = df.drop(columns=['types', 'all_reviews', 'desc_snippet', 'recent_reviews', 'developer',
                       'publisher', 'achievements', 'mature_content', 'minimum_requirements',
                       'recommended_requirements', 'discount_price'])

**Comma-separated columns** — add a space after each comma so the text reads naturally once it's fed into the embedding

In [ ]:
for col in ['genre', 'popular_tags', 'game_details', 'languages']:
    df[col] = df[col].str.replace(',', ', ', regex=False)

**Release date** — drop rows without a real year, then normalize whatever's left down to a single 4-digit year

In [ ]:
df = df[df['release_date'].str.contains(r'\d{4}', na=False)]
df['release_date'] = df['release_date'].str.replace(r'(\d{4})\d{4}', r'\1', regex=True).str.findall(r'\d{4}').str[-1]
df['release_date'] = df['release_date'].str.extract(r'(\d{4})(?!.*\d{4})')
df['release_date'] = df['release_date'].astype(int)

**App ID** — extracted from the Steam store URL, needed to merge with the second dataset and to call the Steam API later

In [ ]:
df['appid'] = df['url'].str.extract(r'/app/(\d+)')

**Description** — strip markdown asterisks and collapse whitespace (this is exactly the noise we saw in section 3)

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return text
    text = re.sub(r"\*+|\s+", ' ', text)
    text = text.lower()
    return text.strip()

df['game_description'] = df['game_description'].apply(clean_text)

**Price** — strip the `$`, coerce to numeric, treat `Free` as `0`

In [ ]:
df['original_price'] = df['original_price'].replace({'Free': '0', '0': '0'})
df['original_price'] = df['original_price'].astype(str).str.replace('$', '', regex=False)
df['original_price'] = pd.to_numeric(df['original_price'], errors='coerce')
df['original_price'] = df['original_price'].fillna(0.0)

### Merge with the app-details dataset

The second dataset adds `header_image`, `required_age` and `metacritic_score` — all keyed by `appid`.

In [ ]:
df1 = df1[['header_image', 'required_age', 'short_description', 'appid', 'metacritic_score']].copy()

# '17+' -> 17, invalid values (e.g. leftover JS from scraping) -> 0
df1['required_age'] = df1['required_age'].astype(str).str.replace('+', '', regex=False)
df1['required_age'] = pd.to_numeric(df1['required_age'], errors='coerce').fillna(0).astype(int)
df1['required_age'] = df1['required_age'].apply(
    lambda x: "For all ages" if x == 0 else f"Game for people over {x} years old"
)

df['appid'] = df['appid'].astype(str)
df1['appid'] = df1['appid'].astype(str)

df = pd.merge(df, df1, on='appid', how='inner')
df.head(3)

## 5. Save the cleaned dataset

`02_feature_engineering.ipynb` picks up from here to build the embedding text and `quality_score`.

In [ ]:
df.to_pickle('../game_on/data/df_clean_step1.pkl')
print(f"Saved {len(df)} cleaned games")